In [ ]:
# --- STEP 1: INSTALL LIBRARIES ---
!pip install yfinance pandas --quiet

import yfinance as yf
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# --- STEP 2: CONFIGURATION ---
LOOKBACK_DAYS = 60      # Range calculation (Quarterly Cycle)
DISCOUNT_LEVEL = 0.20   # Bottom 20%
PREMIUM_LEVEL = 0.80    # Top 20%
EARNINGS_BUFFER = 21    # Days to avoid before earnings

# 4H Trend Settings
FAST_EMA = 5 
SLOW_EMA = 20

# Your Watchlist
tickers = [
    "A", "AA", "AAL", "AAP", "AAPL", "ABBV", "ABNB", "ABT", "ACN", "ADBE", "ADI", "ADM", "ADP", "ADSK", "AEP", "AES", "AFL", "AFRM", "AGNC", "AI", "AIG", "AKAM", "ALB", "ALK", "ALL", "AMAT", "AMD", "AMGN", "AMRN", "AMZN", "APA", "APH", "APTV", "AVGO", "AXP", "BA", "BABA", "BAC", "BAX", "BBY", "BEN", "BIDU", "BIIB", "BITO", "BK", "BKR", "BMY", "BP", "BSX", "BX", "BYND", "C", "CAH", "CAT", "CB", "CCI", "CCJ", "CCL", "CF", "CFG", "CHWY", "CI", "CL", "CLF", "CLX", "CMCSA", "CME", "CNC", "CNP", "COF", "COIN", "COP", "COST", "CPB", "CPRI", "CRM", "CRON", "CRWD", "CSCO", "CSX", "CTVA", "CVNA", "CVS", "CVX", "CZR", "D", "DAL", "DD", "DE", "DELL", "DHI", "DHR", "DIA", "DIS", "DKNG", "DLR", "DOCU", "DOW", "DRI", "DVN", "DXC", "EA", "EBAY", "ED", "EEM", "EFA", "EIX", "EL", "EMR", "ENPH", "EOG", "EQR", "EQT", "ETSY", "EVRG", "EW", "EWJ", "EWW", "EWY", "EWZ", "EXC", "EXPE", "F", "FANG", "FAST", "FCX", "FDX", "FE", "FI", "FITB", "FOXA", "FSLR", "FTI", "FTV", "FXE", "FXI", "GD", "GDX", "GE", "GILD", "GLD", "GLW", "GM", "GME", "GOOG", "GOOGL", "GPRO", "GPS", "GS", "HAL", "HBAN", "HBI", "HCA", "HD", "HIG", "HLT", "HOG", "HOLX", "HON", "HPE", "HPQ", "HYG", "IBB", "IBIT", "IBM", "ICE", "IEF", "INTC", "IP", "IPG", "IRM", "IVZ", "IWM", "IYR", "JCI", "JD", "JETS", "JNJ", "JNPR", "JPM", "K", "KHC", "KMI", "KO", "KR", "KRE", "KSS", "KWEB", "LEN", "LI", "LKQ", "LLY", "LNC", "LOW", "LQD", "LUMN", "LUV", "LVS", "LYB", "LYFT", "MA", "MAR", "MARA", "MAS", "MCD", "MCHP", "MDLZ", "MDT", "MET", "META", "MGM", "MMM", "MO", "MOS", "MPC", "MRK", "MRNA", "MRVL", "MS", "MSFT", "MSTR", "MTB", "MTCH", "MU", "NCLH", "NEE", "NEM", "NET", "NFLX", "NKE", "NOW", "NRG", "NTAP", "NTES", "NVAX", "NVDA", "NWL", "NWSA", "O", "OIH", "OKE", "OMC", "ON", "ORCL", "OXY", "PARA", "PBR", "PDD", "PENN", "PEP", "PFE", "PFG", "PG", "PGR", "PINS", "PLTR", "PNC", "PPL", "PRU", "PSX", "PTON", "PYPL", "QCOM", "QQQ", "RBLX", "RCL", "RF", "RIG", "RIOT", "RIVN", "ROKU", "ROST", "RTX", "RUN", "SBUX", "SCHD", "SCHW", "SEDG", "SHOP", "SIRI", "SLB", "SLV", "SMCI", "SMH", "SNAP", "SNOW", "SO", "SOFI", "SOXL", "SOXS", "SPG", "SPX", "SPXL", "SPXS", "SPY", "SQQQ", "STX", "SWKS", "SYF", "SYY", "T", "TAP", "TBT", "TCOM", "TDOC", "TFC", "TGT", "TJX", "TLT", "TMO", "TMUS", "TPR", "TQQQ", "TRIP", "TRV", "TSLA", "TSM", "TSN", "TT", "TTD", "TTWO", "TXN", "U", "UA", "UAA", "UAL", "UBER", "ULTA", "UNG", "UNH", "UNM", "UNP", "UPS", "UPST", "URBN", "USB", "USO", "UVXY", "V", "VALE", "VFC", "VGK", "VTR", "VXX", "VZ", "WAB", "WBA", "WBD", "WDC", "WFC", "WM", "WMB", "WMT", "WU", "WY", "WYNN", "X", "XBI", "XEL", "XHB", "XLB", "XLC", "XLE", "XLF", "XLI", "XLK", "XLP", "XLRE", "XLU", "XLV", "XLY", "XOM", "XOP", "XRT", "XRX", "XSP", "YELP", "YINN", "ZION", "ZM"
]
tickers = list(set(tickers))

# --- STEP 3: FUNCTIONS ---

def calculate_ema(series, span):
    return series.ewm(span=span, adjust=False).mean()

def check_earnings_and_link(ticker):
    """
    Checks if earnings are within 21 days and returns a Yahoo link.
    """
    status = "Check Link"
    link = f"https://finance.yahoo.com/quote/{ticker}"
    
    try:
        stock = yf.Ticker(ticker)
        cal = stock.calendar
        earnings_date = None
        
        if isinstance(cal, dict) and 'Earnings Date' in cal:
             earnings_date = cal['Earnings Date'][0]
        elif isinstance(cal, pd.DataFrame) and not cal.empty:
             earnings_date = cal.iloc[0, 0]
        
        if earnings_date:
            earnings_date = pd.to_datetime(earnings_date).replace(tzinfo=None)
            today = datetime.now().replace(tzinfo=None)
            days_until = (earnings_date - today).days
            
            if 0 <= days_until <= EARNINGS_BUFFER:
                status = f"WARNING ({days_until} days)"
            else:
                status = "SAFE"
    except:
        pass 
        
    return status, link

# --- STEP 4: MAIN SCANNER ---

def scan_market(ticker_list):
    print(f"Scanning {len(ticker_list)} assets...")
    daily_data = yf.download(ticker_list, period="6mo", interval="1d", group_by='ticker', progress=False)
    hourly_data = yf.download(ticker_list, period="1mo", interval="1h", group_by='ticker', progress=False)

    opportunities = []

    for ticker in ticker_list:
        try:
            # --- MACRO ---
            df_daily = daily_data[ticker].dropna()
            if len(df_daily) < LOOKBACK_DAYS: continue

            df_recent = df_daily.tail(LOOKBACK_DAYS)
            current_price = df_recent['Close'].iloc[-1]
            period_high = df_recent['High'].max()
            period_low = df_recent['Low'].min()

            if period_high == period_low: continue
            position = (current_price - period_low) / (period_high - period_low)
            if position > DISCOUNT_LEVEL and position < PREMIUM_LEVEL: continue

            # --- TREND ---
            df_hourly = hourly_data[ticker].dropna()
            if len(df_hourly) < 20: continue
            df_4h = df_hourly.resample('4h').agg({'Open': 'first', 'High': 'max', 'Low': 'min', 'Close': 'last'}).dropna()
            df_4h['EMA_5'] = calculate_ema(df_4h['Close'], FAST_EMA)
            df_4h['EMA_20'] = calculate_ema(df_4h['Close'], SLOW_EMA)
            last_close = df_4h['Close'].iloc[-1]
            ema_5 = df_4h['EMA_5'].iloc[-1]
            ema_20 = df_4h['EMA_20'].iloc[-1]

            signal = "None"
            strike_target = 0
            
            if position <= DISCOUNT_LEVEL:
                if (ema_5 > ema_20) and (last_close > ema_20):
                    signal = "BUY"
                    strike_target = period_low 

            elif position >= PREMIUM_LEVEL:
                if (ema_5 < ema_20) and (last_close < ema_20):
                    signal = "SELL"
                    strike_target = period_high

            if signal != "None":
                e_status, e_link = check_earnings_and_link(ticker)
                
                opportunities.append({
                    'Ticker': ticker,
                    'Signal': signal,
                    'Price': round(current_price, 2),
                    'Rec_Strike': round(strike_target, 2),
                    'Earnings': e_status,
                    'Link': e_link
                })

        except Exception as e:
            continue

    return pd.DataFrame(opportunities)

# --- EXECUTE ---
results = scan_market(tickers)

print("\n" + "="*80)
print(f" TIME FREEDOM SCANNER - {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')} EST")
print("="*80)

if not results.empty:
    results = results.sort_values(by=['Signal'])
    
    # 1. DISPLAY FULL TABLE (For Research)
    cols = ['Ticker', 'Signal', 'Price', 'Rec_Strike', 'Earnings', 'Link']
    print(results[cols].to_string(index=False))
    
    # 2. FILTERED WATCHLIST (SAFE ONLY)
    print("\n" + "-"*80)
    print(" WATCHLIST EXPORT (TRADINGVIEW) - ONLY 'SAFE' STOCKS COPIED BELOW")
    print("-" * 80)
    
    safe_buys = results[(results['Signal'] == "BUY") & (results['Earnings'] == "SAFE")]['Ticker'].tolist()
    safe_sells = results[(results['Signal'] == "SELL") & (results['Earnings'] == "SAFE")]['Ticker'].tolist()
    
    if safe_buys:
        print(f"COPY BUYS: {','.join(safe_buys)}")
    else:
        print("COPY BUYS: (None found without Earnings risk)")
        
    if safe_sells:
        print(f"COPY SELLS: {','.join(safe_sells)}")
    else:
        print("COPY SELLS: (None found without Earnings risk)")
    
else:
    print("\nNo setups found right now.")

# --- RULES ---
print("\n" + "="*80)
print(" EXECUTION RULES (CASINO MATH)")
print("="*80)
print("1. EXPIRY DATE:")
print("   • Select Expiration: 21 to 30 Days Out.")
print("-" * 80)
print("2. MINIMUM REWARD REQUIREMENT:")
print("   • Make sure your Reward (Credit) is greater than 50% of your Risk.")
print("   • (Example: If you Risk $200, you must collect at least $100 Credit).")
print("-" * 80)
print("3. EARNINGS:")
print("   • The Watchlist above AUTOMATICALLY filtered out 'Earnings Warnings'.")
print("   • If you trade a stock from the main table manually, CHECK THE LINK.")
print("="*80)